In [ ]:
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import gc
import os

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

In [ ]:
class BitFlipOptimizer(torch.optim.Optimizer):
    def __init__(self, params, lr=0.01, momentum=0.9, num_flips=1):
        defaults = dict(lr=lr, momentum=momentum, num_flips=num_flips)
        super().__init__(params, defaults)
        for group in self.param_groups:
            for p in group['params']:
                self.state[p] = {}
                
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                if not getattr(p, 'is_baseline_binary', False):
                    p.data -= group['lr'] * p.grad.data
                    continue
                if "momentum_buffer" not in self.state[p]:
                    self.state[p]["momentum_buffer"] = torch.zeros_like(p.grad.data)
                m = self.state[p]["momentum_buffer"]
                m = group["momentum"] * m + (1 - group["momentum"]) * p.grad.data
                self.state[p]["momentum_buffer"] = m
                scores = m * p.data
                _, min_wi = torch.topk(scores.view(-1), min(group["num_flips"], p.shape[0] * p.shape[1]), largest=False)
                w = p.data.view(-1)
                scores = scores.view(-1)
                for i in min_wi:
                    if scores[i] < 0:
                        w[i] *= -1.0
        return loss

In [ ]:
class BinaryLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        W = torch.randn(out_features, in_features, device=device)
        W_borth = torch.sign(self.newtonschulz5(W))
        self.weight = nn.Parameter(W_borth, requires_grad=False)
        self.bias = nn.Parameter(torch.zeros(out_features, device=device))
        self.weight.is_baseline_binary = True

    def forward(self, x):
        return x @ self.weight.T + self.bias

    def newtonschulz5(self, G, steps=5, eps=1e-7):
        assert G.ndim == 2
        a, b, c = (3.4445, -4.7750, 2.0315)
        X = G.float()
        X /= (X.norm() + eps)
        if G.size(0) > G.size(1):
            X = X.T
        for _ in range(steps):
            A = X @ X.T
            B = b * A + c * A @ A
            X = a * X + B @ X
        if G.size(0) > G.size(1):
            X = X.T
        return X

In [ ]:
class mnist(nn.Module):
    def __init__(self):
        super().__init__()
        q, _ = torch.linalg.qr(torch.randn(784, 784, device=device))
        self.register_buffer('q', q)
        self.layer1 = BinaryLinear(784, 1024)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.2)
        self.layer2 = BinaryLinear(1024, 1024)
        self.relu2 = nn.ReLU()
        self.classifier = nn.Linear(1024, 10)
        
    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = x @ self.q
        x = self.relu1(self.layer1(x))
        residual = x
        x = self.drop1(x)
        x = self.relu2(self.layer2(x))
        x = x + residual
        return self.classifier(x)

model = mnist().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = BitFlipOptimizer(model.parameters(), lr=0.01, momentum=0.9, num_flips=5)
gc.collect()

In [ ]:
epochs = 25
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []
checkpoint_path = "mnist_checkpoint.pt"
start_epoch = 0

In [ ]:
if os.path.exists(checkpoint_path):
    print(f"Found checkpoint '{checkpoint_path}', resuming training")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_losses = checkpoint['train_losses']
    train_accuracies = checkpoint['train_accuracies']
    test_losses = checkpoint['test_losses']
    test_accuracies = checkpoint['test_accuracies']
    print(f"Resuming from loop index {start_epoch}, epoch {start_epoch + 1}")
else:
    print("No checkpoint found, starting training from scratch.")

In [ ]:
for epoch in range(start_epoch, epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / len(train_loader)))  
    train_losses.append(running_loss / len(train_loader))
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / len(test_loader)))
    test_losses.append(running_loss_eval / len(test_loader))
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)

    checkpoint_data = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'test_losses': test_losses,
        'test_accuracies': test_accuracies
    }
    tmp_checkpoint_path = checkpoint_path + ".tmp"
    torch.save(checkpoint_data, tmp_checkpoint_path)
    os.replace(tmp_checkpoint_path, checkpoint_path)
    print(f"Checkpoint saved securely at end of epoch {epoch + 1}")
    
    gc.collect()